# Module 4 — Image compression

Compare self-contained Huffman, horizontal/vertical RLE and JPEG Q=90/50/10/5 on the original frame, eye region, segmented RGB image and algorithm-generated mask. Ratios include codec metadata. These experiments measure compression effects, not anatomical accuracy.

The full-video Q=50/10 study transforms upright decoded frames before eye tracking and segmentation; pixel dimensions and the original ruler calibration are preserved. Ordinary runs retain selected-frame examples and CSV evidence. Set `module4.save_all_frames: true` to retain every temporal JPEG for a future bundled example.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "config.yaml").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from IPython.display import display, Image
from psib.config import load_config
from psib.compression_study import prepare_study, selected_frame_study, temporal_study
cfg = load_config(ROOT / "config.yaml")
study = prepare_study(cfg)
print("Module 4 outputs:", study["out"])
print("Original calibration (pixels/cm):", study["scale"])

## 4.1 Lossless and JPEG comparisons

Huffman uses an explicitly serialized prefix code. RLE resets at each scanline and traverses RGB channels independently. Horizontal and vertical variants use the same run representation. Exact lossless reconstruction is checked before accepting results. Compression ratio below 1 means expansion; lossless PSNR is infinite and SSIM is 1.

In [ ]:
metrics = selected_frame_study(study)
display(metrics.drop(columns=["sha256", "path"]))
display(Image(filename=str(cfg["_output_dir"] / "figures" / "m4_quality_vs_ratio.png")))
# Compact visual evidence is embedded; all remaining comparisons are in Results.
for index in (0, 1):
    display(Image(filename=str(cfg["_output_dir"] / "figures" / f"m4_frame{index}_eye_rgb_edge.png")))

## 4.2 Whole-video measurement effects

Run an independent uncompressed baseline and JPEG Q=50/10 passes with identical Module 3 settings and one shared stimulus time. Curves retain raw rejected measurements and artifact reasons. MAE/RMSE/bias use mutually accepted raw measurements, with paired counts and rejection counts reported. No accepted pairs means unavailable results. Physiological changes remain exploratory; interpolation is not evidence that a mask is accurate.

In [ ]:
comparison, physiology = temporal_study(study)
display(comparison)
display(physiology)
display(Image(filename=str(cfg["_output_dir"] / "figures" / "m4_temporal_diameter.png")))

## 4.3 Inspect and discuss

Inspect the selected-frame reconstruction, pupil-edge and absolute-error figures in Results. All error images use the same 0–255 scale; RGB errors are averaged over channels.

1. At which JPEG quality do you first see 8×8 blocking in this recording? Compare Q=90/50/10/5 without changing display scale.
2. Where are ringing/block boundaries most evident: pupil edges or uniform iris/sclera? Distinguish artificial edges from zeroing the segmented background.
3. Does Q=50/10 change diameter, acceptance rate or response estimates? Cite paired sample counts, bias, MAE/RMSE and the temporal curves. Missing windows prevent physiological conclusions.
4. Explain differences between horizontal/vertical RLE and Huffman across RGB images and masks. Include complete file overhead and cases where data expands.

The source video is already compressed. This study measures **additional still-image JPEG compression of decoded frames**, not video bitrate or a lossless camera reference. The binary-mask JPEG experiment evaluates image intensities only; it does not threshold JPEG masks and treat them as new segmentations.